# Notebook to check that duplicate sequences are not cluttering storage

In [ ]:
import cogent3
from cogent3 import get_app
from cogent3 import load_aligned_seqs
import paths
import libs
import pandas as pd

In [36]:
@cogent3.app.composable.define_app
def take_names(aln: cogent3.app.typing.AlignedSeqsType) -> pd.DataFrame:
    n_names = len(aln.names)
    df_names = pd.DataFrame({'name': list(aln.names), 'aln_length': [n_names] * len(aln.names)})
    return df_names

In [38]:
chrm = "20/"
region = "introns/alldata_chrm" + chrm
sequence = "ENSG00000125975.fa"
folder_in = paths.DATA_HUMCHIMPORANG115 + region
aln = load_aligned_seqs(filename=folder_in + sequence, moltype='dna')
app_names = take_names()

app_names(aln)

,name,aln_length
0,pongo_abelii:20:31909701-31915588:-1,3
1,pan_troglodytes:20:36962670-36968571:-1,3
2,homo_sapiens:20:35523185-35529652:-1,3


In [ ]:
region = "introns/alldata_chrm" + chrm
folder_in = paths.DATA_HUMCHIMPORANG115 + region
in_dstore = cogent3.open_data_store(folder_in, suffix='fa', mode='r')

app_names = take_names()
results = []

for member in in_dstore:
    try:
        aln = load_aligned_seqs(folder_in + '/' + str(member), moltype='dna')
        results.append(app_names(aln))
    except Exception as e:
        print(f"Skipping {member}: {e}")

allaln_names = pd.concat(results, ignore_index=True) if results else pd.DataFrame(columns=['name', 'aln_length'])
allaln_names

,name,aln_length
0,pan_troglodytes:20:25847806-25850971:-1,3
1,homo_sapiens:20:25612934-25616093:-1,3
2,pongo_abelii:20:21063301-21066696:-1,3
3,pongo_abelii:20:16820800-17848687:-1,3
4,pan_troglodytes:20:23215400-23221775:-1,3
...,...,...
1801,pan_troglodytes:20:13591855-13673724:-1,3
1802,homo_sapiens:20:13714321-13785409:-1,3
1803,homo_sapiens:20:62937059-62948589:1,3
1804,pan_troglodytes:20:65105438-65118249:1,3


Given that only 6 sequences appear duplicated, and that rename removes the alignments they belong to (aln_length>3), duplicates are not a problem for my pipeline. Storage cluttered is more likly to be caused by site redundancy ("?" sites are represented twice) and multiple files storage.

In [46]:
# keep only rows whose name appears more than once
duplicate_names = allaln_names[allaln_names['name'].duplicated(keep=False)]
duplicate_names = duplicate_names.sort_values(by='name')
duplicate_names

,name,aln_length
278,homo_sapiens:20:1578810-1604153:-1,6
985,homo_sapiens:20:1578810-1604153:-1,6
280,homo_sapiens:20:1612546-1635609:-1,6
987,homo_sapiens:20:1612546-1635609:-1,6
277,pan_troglodytes:20:1494269-1519500:-1,6
984,pan_troglodytes:20:1494269-1519500:-1,6
282,pongo_abelii:20:26006873-26035943:1,6
989,pongo_abelii:20:26006873-26035943:1,6
279,pongo_abelii:20:26043244-26066536:-1,6
986,pongo_abelii:20:26043244-26066536:-1,6


When I rename all the data before counting them, only alignments of length 3 are sampled.

In [56]:
region = "introns/alldata_chrm" + chrm
folder_in = paths.DATA_HUMCHIMPORANG115 + region
in_dstore = cogent3.open_data_store(folder_in, suffix='fa', mode='r')

app_names = take_names()
rename_noncds = libs.renamer_noncds_aligned()
results = []

for member in in_dstore:
    try:
        aln = load_aligned_seqs(folder_in + '/' + str(member), moltype='dna')
        renamed_aln = rename_noncds(aln)
        result = app_names(renamed_aln)
        if isinstance(result, pd.DataFrame):
            results.append(result)
        else:
            print(f"Skipping non-DataFrame result for {member}: {type(result)}")
    except Exception as e:
        print(f"Skipping {member}: {e}")

Skipping non-DataFrame result for ENSG00000130702-0.fa: <class 'cogent3.app.composable.NotCompleted'>
Skipping non-DataFrame result for ENSG00000130584-0.fa: <class 'cogent3.app.composable.NotCompleted'>
Skipping non-DataFrame result for ENSG00000184617-1.fa: <class 'cogent3.app.composable.NotCompleted'>
Skipping non-DataFrame result for ENSG00000170367-0.fa: <class 'cogent3.app.composable.NotCompleted'>
Skipping non-DataFrame result for ENSG00000182931-1.fa: <class 'cogent3.app.composable.NotCompleted'>
Skipping non-DataFrame result for ENSG00000101292-1.fa: <class 'cogent3.app.composable.NotCompleted'>
Skipping non-DataFrame result for ENSG00000180383.fa: <class 'cogent3.app.composable.NotCompleted'>
Skipping non-DataFrame result for ENSG00000243509.fa: <class 'cogent3.app.composable.NotCompleted'>
Skipping non-DataFrame result for ENSG00000260861-0.fa: <class 'cogent3.app.composable.NotCompleted'>
Skipping non-DataFrame result for ENSG00000180083.fa: <class 'cogent3.app.composable.N

In [58]:
renamedaln_names = pd.concat(results, ignore_index=True) if results else pd.DataFrame(columns=['name', 'aln_length'])
renamedaln_names

,name,aln_length
0,Human,3
1,Chimpanzee,3
2,Orangutan,3
3,Human,3
4,Chimpanzee,3
...,...,...
1615,Chimpanzee,3
1616,Orangutan,3
1617,Human,3
1618,Chimpanzee,3


In [59]:
renamedaln_names["aln_length"].ne(3).any()

np.False_